# 42 · Ajuste gradual de los codificadores seleccionados

**Revisión 1.1 · Plan metodológico v1.2 · GPU.**

La campaña autorizada comprende **20 trabajos**: **DenseNet121, ViT-B/16, Skelex y MambaOut-Tiny**, cada uno en los mismos cinco pliegues de desarrollo, con semilla 2026. La selección se vuelve a comprobar contra los resultados completos y privados de las libretas 40 y 41; esta libreta no agrega candidatos ni cambia los criterios de selección.

Cada pliegue nuevo parte de los pesos externos registrados y una cabeza nueva: cinco épocas con el codificador congelado, cinco con el último bloque entrenable y después ajuste del codificador completo, hasta el límite y la parada temprana del plan. Se mantienen las particiones por participante, las imágenes de 224 × 224 y el aumento gamma únicamente en entrenamiento.

Antes de ejecutar un modelo pendiente se comprueban su asignación de capas, gradientes y memoria en GPU. Un ajuste autorizado del tamaño de los microlotes conserva el lote efectivo de 16; estas comprobaciones no aportan pesos ni métricas a los pliegues completos.

Los trabajos completos se conservan sin copiar sus checkpoints al entorno de Colab. Los trabajos nuevos guardan el estado necesario al final de cada época para poder recuperarse. Se mantienen las dos generaciones recientes verificadas de cada trabajo; una época interrumpida antes de guardarse se repite. Para retomar un pliegue, conserva esta revisión y el mismo tipo de GPU y entorno; si no son compatibles, la libreta se detiene y conserva el avance. Ejecuta una sola sesión de esta campaña.

Las primeras cuatro etapas preparan y comprueban las entradas. **El entrenamiento comienza al ejecutar la etapa 5**, de acuerdo con la autorización ya otorgada; no se solicita una nueva frase ni es necesario escribir rutas. Los originales y las particiones no se modifican. La prueba reservada, la fusión multimodal, la calibración y el ajuste final quedan fuera de esta campaña.

In [ ]:
from pathlib import Path, PurePosixPath
import json, os, re, shutil, stat, subprocess, sys, tempfile, urllib.request, zipfile

CODE_REVISION = '8f9832fac7e53862f0c16020c82898e5cb6a74e6'
if not re.fullmatch(r'[0-9a-f]{40}', CODE_REVISION) or CODE_REVISION == '0' * 40:
    raise RuntimeError('Esta copia no tiene una revisión publicada. Usa la libreta final entregada.')
if os.environ.get('COLAB_GPU', '0') != '1' or any(os.environ.get(k) for k in ('COLAB_TPU_ADDR', 'TPU_NAME', 'COLAB_TPU_TASK_ID')):
    raise RuntimeError('Selecciona GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución. No uses CPU ni TPU.')
if not os.environ.get('COLAB_RELEASE_TAG'):
    raise RuntimeError('Esta libreta debe ejecutarse personalmente en Google Colab; no es un ensayo local.')
if 'knee' in sys.modules or any(name.startswith('knee.') for name in sys.modules):
    raise RuntimeError('Reinicia el entorno para cargar la revisión fijada de esta libreta.')
SESSION_ROOT = Path(tempfile.mkdtemp(prefix='knee_gradual42_'))
REPO_DIR = SESSION_ROOT / 'repository'
archive_path = SESSION_ROOT / 'code.zip'
url = 'https://codeload.github.com/AlonsoYess/Knee/zip/' + CODE_REVISION
with urllib.request.urlopen(url, timeout=60) as response, archive_path.open('xb') as target:
    if response.geturl() != url:
        raise RuntimeError('La descarga del código cambió de ubicación.')
    total = 0
    while True:
        chunk = response.read(1024**2)
        if not chunk:
            break
        total += len(chunk)
        if total > 128 * 1024**2:
            raise RuntimeError('El archivo del repositorio supera el límite autorizado.')
        target.write(chunk)
REPO_DIR.mkdir()
with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    if not members or len(members) > 20000 or sum(i.file_size for i in members) > 512 * 1024**2:
        raise RuntimeError('El contenido del repositorio supera los límites de extracción.')
    seen = set()
    for item in members:
        name = item.filename.rstrip('/')
        parts = name.split('/')
        if not name or '\\' in name or ':' in name or '\0' in name or any(p in ('', '.', '..') for p in parts) or parts[0] != 'Knee-' + CODE_REVISION or name in seen or stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Se encontró una ruta no permitida en el repositorio.')
        seen.add(name)
        destination = REPO_DIR.joinpath(*parts[1:])
        if item.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        else:
            if len(parts) < 2:
                raise RuntimeError('El archivo del repositorio no tiene una ruta relativa válida.')
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, destination.open('xb') as target:
                shutil.copyfileobj(source, target, length=1024**2)
sys.path.insert(0, str(REPO_DIR / 'src'))
from knee import gradual_campaign_colab as orchestration

def checked_run(arguments, **kwargs):
    result = subprocess.run(arguments, capture_output=True, text=True, **kwargs)
    if result.returncode:
        detail = re.sub(r'https?://\S+', '<url>', (result.stdout + result.stderr)[-12000:])
        detail = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', detail)
        print(detail)
        raise RuntimeError('La comprobación previa de la campaña se detuvo. Conserva el error; no inicies entrenamiento.')
    return result.stdout

print('Revisión exacta del código preparada. No se abrieron datos de participantes.')

CAMPAIGN = 'gradual_four_models'


## 1. Preparar el intérprete GPU aislado

Se conservan las dependencias fijadas y se verifica el recurso seleccionado. Esta celda no inicia entrenamiento.

In [ ]:
print('Preparando el intérprete aislado; todavía no se inició entrenamiento…')
runtime = json.loads(checked_run([sys.executable, '-B', str(REPO_DIR / 'scripts/train_gradual_campaign_colab.py'), '--operation', 'setup', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR), '--destination', str(SESSION_ROOT / 'isolated_runtime')]))
NEURAL_PYTHON = Path(runtime['python_path'])
RUN_ENV = orchestration.runtime_environment(REPO_DIR, CAMPAIGN)
print('Intérprete preparado. Todavía no se entrenó ningún trabajo.')


## 2. Comprobar la implementación con datos sintéticos

Se comprueban el alcance, la selección, la recuperación y la separación entre preparación y entrenamiento. Las pruebas utilizan CPU con la GPU oculta y no abren datos de participantes.

In [ ]:
TEST_ENV = dict(RUN_ENV)
TEST_ENV.update(CUDA_VISIBLE_DEVICES='', COLAB_GPU='0')
CPU_TESTS = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/train_gradual_campaign_colab.py'), '--operation', 'tests', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR)], cwd=REPO_DIR, env=TEST_ENV))
print(f"Pruebas sintéticas aprobadas: {CPU_TESTS['tests_run']}; omitidas: {CPU_TESTS['skipped']}.")


## 3. Reconstruir únicamente las entradas de desarrollo

Las carpetas se leen de la misma configuración privada utilizada en la libreta 41. Se comprueban las partes del paquete y la referencia de entradas. Esta etapa no selecciona datos nuevos ni descarga el paquete reservado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MY_DRIVE_ROOT = Path('/content/drive/MyDrive').resolve()
PROJECT_ROOT = orchestration.checked_path(os.environ.get('KNEE_DATA_ROOT') or MY_DRIVE_ROOT / 'KneeOAI').resolve()
if not PROJECT_ROOT.is_dir() or not PROJECT_ROOT.is_relative_to(MY_DRIVE_ROOT):
    raise RuntimeError('No se encontró la carpeta privada configurada dentro de tu Drive montado.')
SETTINGS_PATH = orchestration.checked_path(PROJECT_ROOT / 'colab_runtime_paths.json')
if not SETTINGS_PATH.is_file() or not 0 < SETTINGS_PATH.stat().st_size <= 16384:
    raise RuntimeError('Falta la configuración privada de las carpetas de Colab.')
settings = orchestration._json(SETTINGS_PATH)
if not isinstance(settings, dict) or set(settings) != {'version', 'records_relative', 'development_relative'} or settings.get('version') != 'knee_colab_runtime_paths_v1.0':
    raise RuntimeError('La configuración privada de carpetas no tiene el formato esperado.')
selected_paths = []
for key in ('records_relative', 'development_relative'):
    relative = settings[key]
    if not isinstance(relative, str) or not relative or len(relative) > 1024 or '\\' in relative or ':' in relative:
        raise RuntimeError('La configuración debe contener rutas relativas válidas.')
    parts = PurePosixPath(relative)
    if parts.is_absolute() or not parts.parts or any(part in ('', '.', '..') for part in relative.split('/')):
        raise RuntimeError('La configuración no puede salir de la carpeta del proyecto.')
    selected = orchestration.checked_path(PROJECT_ROOT.joinpath(*parts.parts)).resolve()
    if not selected.is_dir() or not selected.is_relative_to(PROJECT_ROOT):
        raise RuntimeError('Una de las carpetas privadas configuradas no existe dentro del proyecto.')
    selected_paths.append(selected)
PERSISTENT_ROOT, REMOTE_PACKAGE_DIR = selected_paths
if PERSISTENT_ROOT.is_relative_to(REMOTE_PACKAGE_DIR) or REMOTE_PACKAGE_DIR.is_relative_to(PERSISTENT_ROOT):
    raise RuntimeError('Las carpetas de registros y desarrollo deben estar separadas.')
if not orchestration.checked_path(REMOTE_PACKAGE_DIR / 'transport.json').is_file():
    raise RuntimeError('La carpeta de desarrollo configurada no contiene transport.json.')
print('Carpetas privadas localizadas automáticamente. No se inició entrenamiento.')
LOCAL_PACKAGE_DIR = SESSION_ROOT / 'development'
REFERENCE_DIR = SESSION_ROOT / 'reference'
REFERENCE_DIR.mkdir(exist_ok=False)
LOCAL_REFERENCE = REFERENCE_DIR / 'model_inputs_loading_reference_v1.json'
transport = orchestration.stage_development(REMOTE_PACKAGE_DIR, LOCAL_PACKAGE_DIR, LOCAL_REFERENCE)
print('Paquete de desarrollo reconstruido y verificado. No se inició entrenamiento.')


## 4. Verificar la selección y preparar los 20 trabajos

Se localizan los resultados completos de las libretas 40 y 41, se comprueban las predicciones y métricas de desarrollo y se reconstruye la selección conforme al plan. Se reconoce el cierre autorizado aunque existan carpetas de intentos incompletos anteriores. El conjunto resultante debe ser DenseNet121, ViT-B/16, Skelex y MambaOut-Tiny. Si la evidencia no coincide, la preparación se detiene. Esta etapa no descarga pesos ni entrena; guarda una preparación verificable en Drive.

In [ ]:
if CAMPAIGN != 'gradual_four_models':
    raise RuntimeError('Esta libreta solo prepara la campaña gradual autorizada.')
import uuid
TEST_RECEIPT = SESSION_ROOT / 'synthetic_test_receipt.json'
RUNTIME_RECEIPT = SESSION_ROOT / 'setup_receipt.json'
for receipt_path, payload in ((TEST_RECEIPT, CPU_TESTS), (RUNTIME_RECEIPT, runtime)):
    if receipt_path.exists():
        if orchestration._json(receipt_path) != payload:
            raise RuntimeError('El recibo pertenece a otra preparación; inicia una sesión nueva.')
    else:
        orchestration._save(receipt_path, payload)
PREPARATION_DIR = SESSION_ROOT / ('gradual_preparation_' + uuid.uuid4().hex)
print('Comprobando las fuentes 40 y 41 y la selección de desarrollo. Sin entrenamiento…')
PREPARATION_CHECK = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/train_gradual_campaign_colab.py'), '--operation', 'prepare', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR), '--records-root', str(PERSISTENT_ROOT), '--package-dir', str(LOCAL_PACKAGE_DIR), '--reference-path', str(LOCAL_REFERENCE), '--output-dir', str(PREPARATION_DIR), '--code-revision', CODE_REVISION, '--test-receipt', str(TEST_RECEIPT), '--runtime-receipt', str(RUNTIME_RECEIPT)], cwd=REPO_DIR, env=RUN_ENV))
orchestration.validate_preparation_receipt(REPO_DIR, PREPARATION_DIR, code_revision=CODE_REVISION)
print('Preparación verificada y guardada. Esta etapa no ejecutó entrenamiento.')


## 5. Ejecutar el ajuste gradual autorizado

Esta celda inicia la comprobación GPU y los trabajos pendientes. Se conserva la receta aprobada y cada pliegue mantiene sus participantes separados. El progreso muestra modelo, pliegue y época; los guardados permiten recuperar las épocas completas después de una interrupción. Los resultados se guardan en la misma carpeta privada de registros.

In [ ]:
if CAMPAIGN != 'gradual_four_models':
    raise RuntimeError('Esta libreta no autoriza otra campaña.')
orchestration.validate_preparation_receipt(REPO_DIR, PREPARATION_DIR, code_revision=CODE_REVISION)
import queue, threading, time, uuid

def training_run(arguments, *, cwd, env):
    # Forward actual stderr progress; stdout contains the final structured record.
    process = subprocess.Popen(arguments, cwd=cwd, env=env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    messages = queue.Queue()
    def forward(stream, channel):
        try:
            for line in iter(stream.readline, ''):
                messages.put((channel, line))
        finally:
            stream.close()
            messages.put((channel, None))
    for stream, channel in ((process.stdout, 'stdout'), (process.stderr, 'stderr')):
        threading.Thread(target=forward, args=(stream, channel), daemon=True).start()
    started, last_heartbeat, closed = time.monotonic(), time.monotonic(), set()
    stdout_lines = []
    try:
        while len(closed) < 2:
            try:
                channel, line = messages.get(timeout=1)
            except queue.Empty:
                channel, line = None, None
            if channel is not None:
                if line is None:
                    closed.add(channel)
                elif channel == 'stdout':
                    stdout_lines.append(line)
                else:
                    safe = re.sub(r'https?://\S+', '<url>', line.rstrip())
                    safe = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', safe)
                    print(safe, flush=True)
            now = time.monotonic()
            if now - last_heartbeat >= 30 and process.poll() is None:
                print(f'Subproceso activo; tiempo transcurrido: {int(now - started)} s. No es un porcentaje de avance.', flush=True)
                last_heartbeat = now
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if status:
        raise RuntimeError('El entrenamiento se detuvo. Conserva el error y los trabajos completos guardados en Drive; no cambies la receta.')
    return ''.join(stdout_lines)

WEIGHTS_ROOT = SESSION_ROOT / 'verified_public_weights'
WEIGHTS_ROOT.mkdir(exist_ok=True)
LOCAL_OUTPUT = SESSION_ROOT / ('gradual_campaign_' + uuid.uuid4().hex)
print('Iniciando solo los trabajos pendientes del ajuste gradual autorizado…', flush=True)
training_log = training_run([str(NEURAL_PYTHON), '-u', '-B', str(REPO_DIR / 'scripts/train_gradual_campaign_colab.py'), '--operation', 'train', '--campaign', CAMPAIGN, '--repository-root', str(REPO_DIR), '--preparation-dir', str(PREPARATION_DIR), '--package-dir', str(LOCAL_PACKAGE_DIR), '--reference-path', str(LOCAL_REFERENCE), '--output-dir', str(LOCAL_OUTPUT), '--records-root', str(PERSISTENT_ROOT), '--weights-root', str(WEIGHTS_ROOT), '--code-revision', CODE_REVISION], cwd=REPO_DIR, env=RUN_ENV)
FINAL_CHECK = json.loads((LOCAL_OUTPUT / orchestration.SUMMARY_FILENAME).read_text(encoding='utf-8'))
orchestration.validate_result(FINAL_CHECK)
print('Campaña gradual terminada y guardada en Drive. La prueba reservada permanece cerrada.')


## 6. Mostrar el resultado

Comparte únicamente el JSON final. Las predicciones individuales, los modelos y los estados de recuperación permanecen privados. Las métricas corresponden a desarrollo y no sustituyen la evaluación final del conjunto reservado.

In [ ]:
print(json.dumps(FINAL_CHECK, ensure_ascii=False, indent=2))
